<div style="text-align: center; padding: 20px;">
<h1 align="center" style="font-size: 28px; color:rgb(156, 90, 255, 1); width: 100%;">━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>تمرین 5 - بازیابی هوشمند اطلاعات
       
موضوع تمرین : رنک کردن با استفاده از شبکه های عصبی
<br>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</h1>

<div style="border: 2px dashed rgba(156, 90, 255, 1); border-radius: 8px; padding: 20px; margin: 20px auto; max-width: 500px; text-align: right;">
<p style="color: rgba(156, 90, 255, 1); font-size: 18px; margin-bottom: 15px;"> مشخصات دانشجو:</p>
<p style="color: #666; margin: 5px;">نام و نام خانوادگی: سهیل حاجیان منش</p>
<p style="color: #666; margin: 5px;">شماره دانشجویی: 810100119</p>
</div>
</div>

<div dir="rtl" style="text-align: justify;  padding: 25px; background-color:rgba(60, 4, 82, 0.21); border-radius: 12px; border: 2px solidrgb(2, 34, 22);"
<div style="line-height: 2.0; font-size: 17px;">
<div style="padding-right:40px">
</div>
<br>

<div style="padding-right:100px">
 ساختار تمرین:     

<b>بخش اول - مجموعه داده </b>

<b>بخش دوم - بازیابی با استفاده از مدل های پیش آموزش دیده شده </b>

<b> بخش سوم - بازیابی با استفاده از مدل های تنظیم دقیق شده </b>

<b> بخش چهارم - آموزش مدل ها با استفاده از تنظیم دقیق کارآمد پارامتری </b>

<b> بخش پنج - ارزیابی جامع مدل ها </b>

<b> بخش ششم - رنک کردن Hybrid </b>

<b> بخش هفتم - پیاده سازی یک RERANKER مبتنی بر LLM با استفاده از GPT-5.1-mini API </b>

</div>
</div>


</div>
</div>

In [2]:
!pip install -q beir datasets sentence-transformers
!pip install -q rank-bm25
!pip install -q peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 17.1 MB/s eta 0:00:00


In [66]:
# ===== Environment Setup =====

import math
import random
import numpy as np
import pandas as pd
import torch
import re
import os
import time


from beir import util
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
from torch.utils.data import DataLoader
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample, losses
from sklearn.metrics.pairwise import cosine_similarity
from peft import LoraConfig, get_peft_model


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


# <div style="text-align: center; direction: rtl;"><h1 align="center" style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>بخش اول - مجموعه داده <br>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</h1></div>


In [4]:
dataset_name = "nfcorpus"
dataset_path = util.download_and_unzip(
    url="https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip",
    out_dir="datasets",
)

print("Dataset path:", dataset_path)

datasets/nfcorpus.zip:   0%|          | 0.00/2.34M [00:00<?, ?iB/s]

Dataset path: datasets/nfcorpus


In [5]:
corpus, queries, qrels = GenericDataLoader(data_folder=dataset_path).load(split="train")

corpus_val, queries_val, qrels_val = GenericDataLoader(data_folder=dataset_path).load(split="dev")

corpus_test, queries_test, qrels_test = GenericDataLoader(data_folder=dataset_path).load(split="test")

print(f"#Documents (corpus): {len(corpus)}")
print(f"#Queries (train): {len(queries)}")
print(f"#Queries (val): {len(queries_val)}")
print(f"#Queries (test): {len(queries_test)}")

  0%|          | 0/3633 [00:00<?, ?it/s]

  0%|          | 0/3633 [00:00<?, ?it/s]

  0%|          | 0/3633 [00:00<?, ?it/s]

#Documents (corpus): 3633
#Queries (train): 2590
#Queries (val): 324
#Queries (test): 323


In [6]:
# Sample document
sample_doc_id = next(iter(corpus))
print("Document ID:", sample_doc_id)
print("Title:", corpus[sample_doc_id]["title"][:100])
print("Text:", corpus[sample_doc_id]["text"][:300])

# Sample query
sample_query_id = next(iter(queries))
print("\nQuery ID:", sample_query_id)
print("Query Text:", queries[sample_query_id])

# Sample qrel
print("\nQrels for this query:", qrels[sample_query_id])

Document ID: MED-10
Title: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland
Text: Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a populati

Query ID: PLAIN-3
Query Text: Breast Cancer Cells Feed on Cholesterol

Qrels for this query: {'MED-2436': 1, 'MED-2437': 1, 'MED-2438': 1, 'MED-2439': 1, 'MED-2440': 1, 'MED-2427': 1, 'MED-2428': 1, 'MED-2429': 1, 'MED-2430': 1, 'MED-2431': 1, 'MED-2432': 1, 'MED-2434': 1, 'MED-2435': 1, 'MED-3238': 1, 'MED-3697': 1, 'MED-3699': 1, 'MED-3241': 1, 'MED-3841': 1, 'MED-3842': 1, 'MED-3843': 1, 'MED-3844': 1, 'MED-3845': 1, 'MED-3846': 1, 'MED-3847': 1, 'MED-3848': 1, 'MED-3849': 1, 'MED-3850': 1, 'MED-4094': 1, 'MED-3852': 1, 'MED-3853': 1, 'MED-3854': 1, 'MED-3855': 1, 'MED-3856': 1

In [7]:
relevance_scores = []

for qid in qrels:
    for doc_id, score in qrels[qid].items():
        relevance_scores.append(score)

relevance_scores = np.array(relevance_scores)

print("Min relevance:", relevance_scores.min())
print("Max relevance:", relevance_scores.max())
print("Total qrels:", len(relevance_scores))

Min relevance: 1
Max relevance: 1
Total qrels: 110575


<div style="text-align: right; direction: rtl;">
درجه بندی ارتباط نداریم و یک سند اگر در qrel آمده باشد مرتبط است و آمده باشد درجه ارتباط آن 1 و مرتبط به حساب می آید.

In [8]:
def build_document_text(doc):
    if doc["title"]:
        return doc["title"] + " " + doc["text"]
    return doc["text"]


corpus_texts = {doc_id: build_document_text(doc) for doc_id, doc in corpus.items()}

In [9]:
print("Sample document text:", corpus_texts[sample_doc_id][:300])

Sample document text: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains uncl


# <div style="text-align: center; direction: rtl;"><h1 align="center" style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>بخش دوم - بازیابی با استفاده از مدل های پیش آموزش دیده شده <br>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</h1></div>


# <div style="text-align: center; direction: rtl;"><h1 style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━<br>الف - بازیابی بدون آموزش به کمک BM25 <br></h1></div>


In [10]:
def bm25_tokenizer(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

In [11]:
doc_ids = list(corpus_texts.keys())
tokenized_corpus = [bm25_tokenizer(corpus_texts[doc_id]) for doc_id in doc_ids]

bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

In [12]:
def bm25_retrieve(query: str, top_k: int = 10):
    tokenized_query = bm25_tokenizer(query)
    scores = bm25.get_scores(tokenized_query)

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in ranked_indices:
        results.append(
            {
                "doc_id": doc_ids[idx],
                "score": float(scores[idx]),
                "text": corpus_texts[doc_ids[idx]],
            }
        )
    return results

In [13]:
sample_qid = next(iter(queries))
sample_query = queries[sample_qid]

bm25_results = bm25_retrieve(sample_query, top_k=5)

print("Query:", sample_query)
print("-" * 60)

for rank, res in enumerate(bm25_results, 1):
    print(f"Rank {rank}")
    print("Doc ID:", res["doc_id"])
    print("Score:", res["score"])
    print("Text snippet:", res["text"][:200])
    print("-" * 60)

Query: Breast Cancer Cells Feed on Cholesterol
------------------------------------------------------------
Rank 1
Doc ID: MED-2434
Score: 19.19214906789834
Text snippet: High ACAT1 expression in estrogen receptor negative basal-like breast cancer cells is associated with LDL-induced proliferation. The specific role of dietary fat in breast cancer progression is unclea
------------------------------------------------------------
Rank 2
Doc ID: MED-2427
Score: 17.271419334080534
Text snippet: Elevated Levels of Cholesterol-Rich Lipid Rafts in Cancer Cells Are Correlated with Apoptosis Sensitivity Induced by Cholesterol-Depleting Agents Lipid rafts/caveolae are membrane platforms for signal
------------------------------------------------------------
Rank 3
Doc ID: MED-3447
Score: 14.70241467371098
Text snippet: Seaweed prevents breast cancer? To investigate the chemopreventive effects of seaweed on breast cancer, we have been studying the relationship between iodine and breast cancer. W

<div style="text-align: right; direction: rtl;">

در نمونه آورده شده سوال در ارتباط با `رابطه بین سلول های سرطان سینه و کلسترول` می باشد.     

سند شماره یک ارتباط بسیار زیادی دارد چون در ارتباط با تغذیه سلول‌های سرطانی از کلسترول است.      
اما بطور مثال سند شماره 4 ارتباط زیادی به پرس و جو ندارد چون سند درباره سرطان پرستات و اثرات هورمونی شیر است.

In [14]:
def bm25_run(queries_dict, top_k=100):
    run = {}
    for qid, query_text in queries_dict.items():
        results = bm25_retrieve(query_text, top_k=top_k)
        run[qid] = {res["doc_id"]: res["score"] for res in results}
    return run

bm25_run_test = bm25_run(queries_test, top_k=100)

<div style="text-align: right; direction: rtl;">
خروجی بالا در محاسبه metric ها در بخش 5 استفاده می شود.

# <div style="text-align: center; direction: rtl;"><h1 style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━<br>ب- بازیابی متراکم مبتنی بر bi-encoder<br></h1></div>


In [15]:
model_name = "sentence-transformers/msmarco-distilbert-base-tas-b"
bi_encoder = SentenceTransformer(model_name, device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
doc_ids = list(corpus_texts.keys())
documents = [corpus_texts[doc_id] for doc_id in doc_ids]

print("Number of documents:", len(documents))

Number of documents: 3633


In [17]:
# ===== Document Embeddings =====

with torch.no_grad():
    doc_embeddings = bi_encoder.encode(
        documents,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

print("Document embeddings shape:", doc_embeddings.shape)

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Document embeddings shape: (3633, 768)


In [18]:
# ===== Query Encoding =====


def encode_query(query_text: str):
    with torch.no_grad():
        q_embedding = bi_encoder.encode(
            query_text,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
    return q_embedding

In [19]:
# ===== Retrieval Function =====


def dense_retrieve(query_text: str, top_k: int = 10):
    q_emb = encode_query(query_text)

    scores = np.dot(doc_embeddings, q_emb)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append(
            {
                "doc_id": doc_ids[idx],
                "score": float(scores[idx]),
                "text": corpus_texts[doc_ids[idx]],
            }
        )
    return results

<div style="text-align: right; direction: rtl;">

چون embedding ها نرمال شده اند cosine similarity دو بردار برابر با dot product آن دو است.

In [20]:
sample_qid = next(iter(queries))
sample_query = queries[sample_qid]

dense_results = dense_retrieve(sample_query, top_k=5)

print("Query:", sample_query)
print("-" * 60)

for rank, res in enumerate(dense_results, 1):
    print(f"Rank {rank}")
    print("Doc ID:", res["doc_id"])
    print("Score:", res["score"])
    print("Text snippet:", res["text"][:200])
    print("-" * 60)

Query: Breast Cancer Cells Feed on Cholesterol
------------------------------------------------------------
Rank 1
Doc ID: MED-2439
Score: 0.7978148460388184
Text snippet: Plant Sterols as Anticancer Nutrients: Evidence for Their Role in Breast Cancer While many factors are involved in the etiology of cancer, it has been clearly established that diet significantly impac
------------------------------------------------------------
Rank 2
Doc ID: MED-2435
Score: 0.7642852067947388
Text snippet: Chemoprevention of breast cancer by dietary compounds. Breast cancer is the leading cause of cancer-related deaths in women in the United States and many other countries. There is an immediate need fo
------------------------------------------------------------
Rank 3
Doc ID: MED-2437
Score: 0.7617143988609314
Text snippet: Diet and breast cancer: understanding risks and benefits. BACKGROUND: Breast cancer is the most commonly diagnosed cancer among women in the United States. Extensive research h

<div style="text-align: right; direction: rtl;">
هم چنان همان سند بی ربط که در BM25 داشتیم در اینجا نیز در رنک 4 وجود دارد.
اما باقی اسناد از نظر "رژیم غذایی، لیپیدها و سرطان پوست" با هم ارتباط دارند.

In [21]:
def dense_run(queries_dict, top_k=100):
    run = {}
    for qid, query_text in queries_dict.items():
        results = dense_retrieve(query_text, top_k=top_k)
        run[qid] = {res["doc_id"]: res["score"] for res in results}
    return run


dense_run_test = dense_run(queries_test, top_k=100)

<div style="text-align: right; direction: rtl;">
خروجی بالا در محاسبه metric ها در بخش 5 استفاده می شود.

<div style="text-align: right; direction: rtl;">

1. **تفاوت بنیادی معماری بین bi-encoders و cross-encoders چیست؟**

    در معماری bi-encoder پرس و جو و سند بصورت جداگانه بعنوان ورودی به یک encoder با وزن یکسان داده می شوند و در نهایت شباهت کسینوسی دو بردار محاسبه می گردد. یعنی پرس و جو و سند مستقل از هم در نظر گرفته می شوند.     

    اما در معماری cross-encoder پرس و جو و سند به صورت همزمان به یک encoder داده می شوند و خروجی encoder یک امتیاز ارتباط بین پرس و جو و سند است. چون هر دو همزمان به encoder داده می شوند، مدل می تواند وابستگی های معنایی و مفهومی پیچیده بین پرس و جو و سند را بهتر بفهمد.

2. **چرا bi-encoders می‌توانند document embeddings را از پیش محاسبه کنند اما cross-encoders نمی‌توانند؟**

    دلیل اصلی به دلیل نوع ورودی در encoder در دو معماری می باشد. در مدل bi-encoder چون سند مستقلا بعنوان ورودی به encoder داده می شود تا بردار embedding آن تولید شود پس می توان تمامی اسناد را از قبل به مدل داد و مقدار embedding آن ها را نگه داشت و در زمان ایجاد پرس و جوها همان بردار از پیش محاسبه شده را برای سند در نظر میگیریم.     
    
    اما cross-encoder چون سند و پرس و جو را باید همزمان به عنوان ورودی به مدل داد، پس نمی توان از پیش محاسبه ای انجام داد و فقط در حین ایجاد پرس و جو باید این مجاسبه را انچام داد.

3. **پیچیدگی محاسباتی بازیابی از بین N سند برای هر رویکرد چیست؟**

    با احتساب متغیر های زیر :
    - N : تعداد اسناد
    - d : ابعاد embedding
    - L: طول doc و query
                    
    `bi-encoder` :  

     1. مرحله آفلاین : O(L.N)
     2. مرحله آنلاین : O(2d.N) -> O(2d) برای ضرب داخلی * N تعداد اسناد  

    
    `cross-encoder` :

    O(L^2 . N) -> O(L^2) برای هر پرس و جو و سند، چون L توکن داریم و همه آنها باهم تعامل دارند.  * N تعداد اسناد

# <div style="text-align: center; direction: rtl;"><h1 style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━<br>پ- بازیابی متراکم مبتنی بر پایه cross-encoder<br></h1></div>


In [22]:
cross_encoder_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(cross_encoder_model_name, device=device)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [23]:
def stage1_bi_encoder(query_text: str, top_k: int = 100):
    results = dense_retrieve(query_text, top_k=top_k)
    return results


def build_cross_encoder_pairs(query_text, candidate_docs):
    pairs = []
    for doc in candidate_docs:
        pairs.append((query_text, doc["text"]))
    return pairs

def stage2_cross_encoder_rerank(query_text, candidates, top_k=10):
    pairs = build_cross_encoder_pairs(query_text, candidates)

    scores = cross_encoder.predict(pairs, batch_size=16, show_progress_bar=False)

    reranked = []
    for doc, score in zip(candidates, scores):
        reranked.append(
            {"doc_id": doc["doc_id"], "score": float(score), "text": doc["text"]}
        )

    reranked = sorted(reranked, key=lambda x: x["score"], reverse=True)
    return reranked[:top_k]

In [24]:
def two_stage_retrieval(query_text, stage1_k=100, final_k=10):
    candidates = stage1_bi_encoder(query_text, top_k=stage1_k)

    final_results = stage2_cross_encoder_rerank(query_text, candidates, top_k=final_k)
    return final_results

In [62]:
def two_stage_retrieve_wrapper(query, top_k=10):
    return two_stage_retrieval(query)

In [25]:
sample_qid = next(iter(queries))
sample_query = queries[sample_qid]

final_results = two_stage_retrieval(sample_query)

print("Query:", sample_query)
print("-" * 60)

for rank, res in enumerate(final_results, 1):
    print(f"Rank {rank}")
    print("Doc ID:", res["doc_id"])
    print("Cross-Encoder Score:", res["score"])
    print("Text snippet:", res["text"][:200])
    print("-" * 60)

Query: Breast Cancer Cells Feed on Cholesterol
------------------------------------------------------------
Rank 1
Doc ID: MED-2434
Cross-Encoder Score: 2.4353854656219482
Text snippet: High ACAT1 expression in estrogen receptor negative basal-like breast cancer cells is associated with LDL-induced proliferation. The specific role of dietary fat in breast cancer progression is unclea
------------------------------------------------------------
Rank 2
Doc ID: MED-2427
Cross-Encoder Score: -1.2546437978744507
Text snippet: Elevated Levels of Cholesterol-Rich Lipid Rafts in Cancer Cells Are Correlated with Apoptosis Sensitivity Induced by Cholesterol-Depleting Agents Lipid rafts/caveolae are membrane platforms for signal
------------------------------------------------------------
Rank 3
Doc ID: MED-5341
Cross-Encoder Score: -1.9192172288894653
Text snippet: Effects of a low-fat, high-fiber diet and exercise program on breast cancer risk factors in vivo and tumor cell growth and apoptosis

<div style="text-align: right; direction: rtl;">

سند بی ربط شماره 2774 که در دو روش قبلی رتبه بالایی داشت این بار رتبه 9 و امتیاز کمی گرفته است.            

از طرفی سند شماره 1 که خیلی ارتباط زیادی نسبت به سایر اسناد دارد هم رتبه اول و امتیاز خیلی بیشتری نسبت به بقیه اسناد گرفته است.
سند شماره اول و دوم در این روش و روش bm25 یکسان شده است. ولی رتبه سوم در این رتبه بندی بسیار مرتبط تر است تا رتبه سوم در bm25.

In [26]:
def two_stage_run(queries_dict, stage1_k=100, final_k=10):
    run = {}
    for qid, query_text in queries_dict.items():
        results = two_stage_retrieval(query_text, stage1_k=stage1_k, final_k=final_k)
        run[qid] = {res["doc_id"]: res["score"] for res in results}
    return run


two_stage_run_test = two_stage_run(queries_test)

<div style="text-align: right; direction: rtl;">
خروجی بالا در محاسبه metric ها در بخش 5 استفاده می شود.

<div style="text-align: right; direction: rtl;">

4. **چرا معمولا خط لوله دومرحله ای استفاده می شود و از cross-encoder تنها استفاده نمی شود؟**

- bi-encoder هزینه محاسباتی کمی دارد و می تواند از میان تعداد زیادی سند، مجموعه اسناد مرتبط را سریع بازیابی کند.
- cross-encoder به دلیل حضور همزمان سند و پرس وجو دقت بالایی دارد اما هزینه محاسباتی آن زیاد است.

بنابر این استفاده تنها از cross-encoder از نظر کارایی قابل مقیاس پذیری نیست.    

به همین دلیل از خط لوله دومرحله استفاده می شود تا سرعت bi-encoder و دقت cross-encoder را همزمان داشته باشیم.

# <div style="text-align: center; direction: rtl;"><h1 align="center" style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>بخش سوم - بازیابی با استفاده از مدل های تنظیم دقیق شده  <br>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</h1></div>


In [27]:
base_model_name = "sentence-transformers/all-MiniLM-L6-v2"
ft_model = SentenceTransformer(base_model_name, device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [28]:
def doc_text(doc_id):
    return corpus_texts[doc_id]

In [29]:
def get_positives(qid, qrels, min_rel=1):
    return [d for d, r in qrels.get(qid, {}).items() if r >= min_rel]

In [83]:
def get_hard_negative(query_text, positives, k=50):
    cands = bm25_retrieve(query_text, top_k=k)

    hard_negs = [
        c["doc_id"]
        for c in cands
        if c["doc_id"] not in positives
    ]

    if not hard_negs:
        return None

    return random.choice(hard_negs)

In [31]:
all_doc_ids = list(corpus_texts.keys())


def get_random_negatives(positives, n=2):
    negs = []
    while len(negs) < n:
        d = random.choice(all_doc_ids)
        if d not in positives:
            negs.append(d)
    return negs

In [84]:
train_examples = []

for qid, qtext in queries.items():
    positives = get_positives(qid, qrels)

    if len(positives) == 0:
        continue

    if len(positives) >= 2:
        selected_positives = random.sample(positives, 2)
    else:
        selected_positives = positives

    for pos_id in selected_positives:
        hard_neg = get_hard_negative(qtext, positives)
        rand_negs = get_random_negatives(positives, n=2)

        negatives = []
        if hard_neg:
            negatives.append(hard_neg)
        negatives.extend(rand_negs)

        for neg_id in negatives:
            train_examples.append(
                InputExample(
                    texts=[
                        qtext,
                        doc_text(pos_id),
                        doc_text(neg_id)
                    ]
                )
            )

print("Triplets:", len(train_examples))

Triplets: 14976


In [85]:
train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [86]:
train_loss = losses.MultipleNegativesRankingLoss(ft_model)

epochs = 3
steps_per_epoch = math.ceil(len(train_loader))
total_steps = steps_per_epoch * epochs
warmup_steps = int(0.1 * total_steps)
learning_rate = 2e-5

In [87]:
os.environ["WANDB_MODE"] = "disabled"

output_dir = "./ft-all-MiniLM-L6-v2"

ft_model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": learning_rate},
    output_path=output_dir,
    save_best_model=False,
    checkpoint_path=output_dir,
    checkpoint_save_steps=steps_per_epoch,
    show_progress_bar=True,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
500,2.468900
1000,2.035700
1500,1.551800
2000,1.360800
2500,1.153600


In [88]:
ft_loaded = SentenceTransformer(output_dir, device=device)

with torch.no_grad():
    ft_doc_emb = ft_loaded.encode(
        [corpus_texts[d] for d in doc_ids],
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def ft_dense_retrieve(query_text, top_k=10):
    q = ft_loaded.encode(query_text, convert_to_numpy=True, normalize_embeddings=True)
    scores = ft_doc_emb @ q
    idx = scores.argsort()[::-1][:top_k]
    return {doc_ids[i]: float(scores[i]) for i in idx}

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

In [89]:
pretrained_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device=device
)

with torch.no_grad():
    pt_doc_emb = pretrained_model.encode(
        [corpus_texts[d] for d in doc_ids],
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def pt_dense_retrieve(query_text, top_k=100):
    q = pretrained_model.encode(
        query_text, convert_to_numpy=True, normalize_embeddings=True
    )
    scores = pt_doc_emb @ q
    idx = scores.argsort()[::-1][:top_k]
    return {doc_ids[i]: float(scores[i]) for i in idx}

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

In [90]:
pretrained_run = {
    qid: pt_dense_retrieve(qtext, top_k=100) for qid, qtext in queries_test.items()
}

In [91]:
finetuned_run = {
    qid: ft_dense_retrieve(qtext, top_k=100) for qid, qtext in queries_test.items()
}

In [92]:
evaluator = EvaluateRetrieval()

metrics_pretrained = evaluator.evaluate(qrels_test, pretrained_run, k_values=[10, 100])

metrics_finetuned = evaluator.evaluate(qrels_test, finetuned_run, k_values=[10, 100])

print("Pre-trained all-MiniLM-L6-v2:", metrics_pretrained)
print("Fine-tuned all-MiniLM-L6-v2:", metrics_finetuned)

Pre-trained all-MiniLM-L6-v2: ({'NDCG@10': 0.31594, 'NDCG@100': 0.29455}, {'MAP@10': 0.11048, 'MAP@100': 0.14244}, {'Recall@10': 0.15499, 'Recall@100': 0.31151}, {'P@10': 0.24334, 'P@100': 0.07981})
Fine-tuned all-MiniLM-L6-v2: ({'NDCG@10': 0.33163, 'NDCG@100': 0.32202}, {'MAP@10': 0.11594, 'MAP@100': 0.16047}, {'Recall@10': 0.16042, 'Recall@100': 0.35197}, {'P@10': 0.26192, 'P@100': 0.09136})


<div style="text-align: right; direction: rtl;">

## همانطور که میبینید بعد از تنظیم دقیق مدل، نتایج تمامی معیارها نسبت به مدل Pre-trained افزایش داشته است.

<div style="text-align: right; direction: rtl;">

**5. مفهوم Hard Negative Mining را توضیح دهید.**

Hard Negative Mining روشی در آموزش مدل است که در آن به‌جای استفاده از نمونه منفی‌های ساده، از نمونه‌های منفیِ سخت استفاده می‌شود؛ یعنی نمونه‌هایی که مدل آن‌ها را به‌اشتباه یا با اطمینان بالا شبیه به نمونه‌های مثبت تشخیص می‌دهد، در حالی که واقعاً منفی هستند.

با اضافه‌کردن این منفی‌های سخت به داده‌ی آموزشی، مدل مجبور می‌شود ویژگی‌های تمایزبخش‌تری یاد بگیرد و بهتر بتواند بین نمونه‌های واقعاً مرتبط  و غیرمرتبط تفاوت قائل شود. این کار معمولاً باعث بهبود دقت مدل، به‌ویژه در رتبه‌بندی و بازیابی اطلاعات، می‌شود.

**6. چرا Hard Negatives برای آموزش مدل های بازیابی اطلاعات مفیدتر از Random Negatives هستند؟**

random negative ها اغلب واضح است که ارتباط ندارند و اکثر مدل ها شرایط سختی برای تشخیص آنها ندارند اما جایی که مدل ها با دشواری بیشتری روبرو هستند و نقطه تمایز مدل ها هم می باشد، رتبه های بالا است و سندهایی که از نظر واژگانی به پرس و جو خیلی نزدیک هستند اما مرتبط نیستند.  قرار دادن این نوع سندها در داده های آموزشی باعث می شود مدل مجبور شود مرزهای دقیق تری یاد بگیرد و کیفیت رتبه های بالاتر بهتر می شود.

**7. اگر در فرایند آموزش تنها از Hard Negatives استفاده کنید، چه مشکلاتی ممکن است برای مدل به وجود آید ؟**

- وقتی مدل بیش از حد بر روی نمونه خاص یادگیری انجام دهد طبیعتا دچار  overfitting می شود و نمی تواند روی داده های آزمون که خیلی اوقات منفی های ساده تری دارند به خوبی عمل کند        

- همگرایی مدل حین آموزش دشوار می شود. چون نمونه های hard negative نوسان شدیدی احتمالا بر روی loss ایجاد می کنند.    

- اگر داده های آموزشی برچسب خورده ایراد داشته باشند و دقیق نباشند ممکن است یک نمونه ای که hard negative تشخیص داده ایم واقعا مرتبط بوده اما برچسب نخورده است یا برچسب اشتباه خورده است.

**8. مفهوم فراموشی فاجعه بار چیست و راه های مقابله با آن چیست ؟**   

فراموشی فاجعه‌بار زمانی رخ می‌دهد که یک مدل یادگیری ماشین هنگام یادگیری وظیفه یا داده‌ی جدید، بخش زیادی از دانش قبلی خود را از دست می‌دهد. یعنی مدل برای عملکرد بهتر روی داده‌های جدید، وزن‌ها را طوری به‌روزرسانی می‌کند که عملکردش روی وظایف قبلی به‌شدت افت می‌کند.
مثلا در حین fine-tune کردن مدل برای یک تسک مشخصی، مدل آموزش هایی که در مرحله pre-trained دیده را فراموش میکند. از نتایج این اتفاق می شه به این اشاره کرد که باید دوباره کلی منابع از جمله هزینه و زمان و منابع طبیعی دیگر مصرف کرد تا دوباره مدل Retrain شود و بتواند task های قدیمی خود را انجام دهد.    
طبق مقاله موجود در لینک https://www.ibm.com/think/topics/catastrophic-forgetting راه های مقابله با catastrophic forgetting :

1. Rehearsal techniques (مرور داده‌های قدیمی) :
    استفاده مجدد از بخشی از داده‌های قبلی (یا نسخه‌ی فشرده‌ای از آن‌ها) هنگام آموزش روی داده‌های جدید، تا مدل دانش قبلی را حفظ کند. Experience replay یک نمونه از همین تکنیک است
2. Regularization-based Methods :
    محدود کردن تغییر وزن‌های مهم برای وظایف قبلی (مثل Elastic weight consolidation) تا مدل نتواند به‌راحتی آن‌ها را تغییر دهد.
3. Parameter Isolation :
    اختصاص پارامترهای جداگانه برای وظایف جدید، به‌طوری‌که پارامترهای مربوط به دانش قبلی دست‌نخورده باقی بمانند.

# <div style="text-align: center; direction: rtl;"><h1 align="center" style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>بخش چهارم - آموزش مدل ها با استفاده از تنظیم دقیق کارآمد پارامتری <br>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</h1></div>


# <div style="text-align: center; direction: rtl;"><h1 style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━<br>الف- پیاده سازی LoRA<br></h1></div>


In [93]:
base_model_name = "sentence-transformers/all-MiniLM-L6-v2"
lora_model = SentenceTransformer(base_model_name, device=device)

In [94]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "key"],
    lora_dropout=0.1,
    bias="none",
    task_type="FEATURE_EXTRACTION",
)

In [95]:
transformer = lora_model._first_module().auto_model

transformer = get_peft_model(transformer, lora_config)

lora_model._first_module().auto_model = transformer

transformer.print_trainable_parameters()

trainable params: 73,728 || all params: 22,786,944 || trainable%: 0.3236


# <div style="text-align: center; direction: rtl;"><h1 style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━<br>ب- آموزش و ذخیره سازی<br></h1></div>


In [96]:
train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)

train_loss = losses.MultipleNegativesRankingLoss(lora_model)

In [97]:
epochs = 3
steps_per_epoch = math.ceil(len(train_loader))
total_steps = steps_per_epoch * epochs
warmup_steps = int(0.1 * total_steps)

os.environ["WANDB_DISABLED"] = "true"

In [100]:
lora_output_dir = "./lora-all-MiniLM-L6-v2"

lora_model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": 2e-5},
    output_path=lora_output_dir,
    save_best_model=False,
    checkpoint_path=lora_output_dir,
    checkpoint_save_steps=steps_per_epoch,
    show_progress_bar=True,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
500,3.111600
1000,3.083800
1500,3.065300
2000,3.039000
2500,3.048300


# <div style="text-align: center; direction: rtl;"><h1 style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━<br>پ - ارزیابی<br></h1></div>


In [101]:
lora_loaded = SentenceTransformer(lora_output_dir, device=device)

with torch.no_grad():
    lora_doc_emb = lora_loaded.encode(
        [corpus_texts[d] for d in doc_ids],
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

In [102]:
def lora_dense_retrieve(query_text, top_k=100):
    q = lora_loaded.encode(query_text, convert_to_numpy=True, normalize_embeddings=True)
    scores = lora_doc_emb @ q
    idx = scores.argsort()[::-1][:top_k]
    return {doc_ids[i]: float(scores[i]) for i in idx}

In [103]:
lora_run_test = {
    qid: lora_dense_retrieve(qtext, top_k=100) for qid, qtext in queries_test.items()
}

In [104]:
evaluator = EvaluateRetrieval()

metrics_lora = evaluator.evaluate(qrels_test, lora_run_test, k_values=[10, 100])

print("Fine-tuned all-MiniLM-L6-v2:", metrics_finetuned)
print("LoRA-adapted model:", metrics_lora)

Fine-tuned all-MiniLM-L6-v2: ({'NDCG@10': 0.33163, 'NDCG@100': 0.32202}, {'MAP@10': 0.11594, 'MAP@100': 0.16047}, {'Recall@10': 0.16042, 'Recall@100': 0.35197}, {'P@10': 0.26192, 'P@100': 0.09136})
LoRA-adapted model: ({'NDCG@10': 0.30578, 'NDCG@100': 0.28237}, {'MAP@10': 0.10705, 'MAP@100': 0.13621}, {'Recall@10': 0.1559, 'Recall@100': 0.29588}, {'P@10': 0.2356, 'P@100': 0.07474})


<div style="text-align: right; direction: rtl;">

نسبت به مدل Fine-tuning عملکرد افت کرده است که کاملا طبق انتظار است و برتری peft در صرفه جویی منابع محاسباتی و هزینه و زمان است.

<div style="text-align: right; direction: rtl;">
تعداد پارامتر های LoRA را با حالت تنظیم دقیق کامل مقایسه کنید:

In [53]:
transformer.print_trainable_parameters()

trainable params: 73,728 || all params: 22,786,944 || trainable%: 0.3236


In [54]:
full_params = sum(
    p.numel()
    for p in ft_loaded._first_module().auto_model.parameters()
    if p.requires_grad
)

full_params

22713216

<div style="text-align: right; direction: rtl;">

## نسبت پارامترهای LoRA نسبت به پارامتر های مدل تنظیم دقیق کامل تنها 0.3% است که نشان می دهد تا چه میزان پارامتر های کمتری را لازم است tune کنیم.

# <div style="text-align: center; direction: rtl;"><h1 align="center" style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>بخش پنج - ارزیابی جامع مدل ها  <br>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</h1></div>


In [105]:
def extract_metrics(metrics):
    ndcg, _map, recall, precision = metrics
    return {
        "MRR@10": _map["MAP@10"],
        "NDCG@10": ndcg["NDCG@10"],
        "Recall@10": recall["Recall@10"],
        "Recall@100": recall.get("Recall@100", None),
        "Precision@10": precision["P@10"]
    }


In [106]:
def measure_latency(retrieve_fn, queries_dict, top_k=10, runs=1):
    times = []
    for _ in range(runs):
        for q in queries_dict.values():
            start = time.time()
            retrieve_fn(q, top_k=top_k)
            times.append(time.time() - start)
    return np.mean(times)

In [107]:
metrics_bm25 = extract_metrics(evaluator.evaluate(qrels_test, bm25_run_test, k_values=[10, 100]))
metrics_pre  = extract_metrics(evaluator.evaluate(qrels_test, pretrained_run, k_values=[10, 100]))
metrics_pipe = extract_metrics(evaluator.evaluate(qrels_test, two_stage_run_test, k_values=[10]))
metrics_ft   = extract_metrics(evaluator.evaluate(qrels_test, finetuned_run, k_values=[10, 100]))
metrics_lora = extract_metrics(evaluator.evaluate(qrels_test, lora_run_test, k_values=[10, 100]))

In [108]:
lat_bm25 = measure_latency(bm25_retrieve, queries_test)
lat_pre  = measure_latency(pt_dense_retrieve, queries_test)
lat_pipe = measure_latency(two_stage_retrieve_wrapper, queries_test)
lat_ft   = measure_latency(ft_dense_retrieve, queries_test)
lat_lora = measure_latency(lora_dense_retrieve, queries_test)


<div style="text-align: right; direction: rtl;">
جدول مربوط به روش های Baseline

In [109]:
table_baselines = pd.DataFrame.from_dict({
    "BM25": {**metrics_bm25, "Mean Latency (s)": lat_bm25},
    "Pre-trained Bi-Encoder": {**metrics_pre, "Mean Latency (s)": lat_pre},
    "Bi-Encoder + Cross-Encoder": {**metrics_pipe, "Mean Latency (s)": lat_pipe},
}, orient="index")

table_baselines


,MRR@10,NDCG@10,Recall@10,Recall@100,Precision@10,Mean Latency (s)
BM25,0.11617,0.30616,0.15222,0.23564,0.21827,0.004057
Pre-trained Bi-Encoder,0.11048,0.31594,0.15499,0.31151,0.24334,0.007657
Bi-Encoder + Cross-Encoder,0.12775,0.33514,0.15114,NaN,0.23282,0.524933


<div style="text-align: right; direction: rtl;">
جدول مربوط به روش های  Trainable

In [110]:
table_trainable = pd.DataFrame.from_dict({
    "Pre-trained": {**metrics_pre, "Mean Latency (s)": lat_pre},
    "Full Fine-Tuning": {**metrics_ft, "Mean Latency (s)": lat_ft},
    "LoRA": {**metrics_lora, "Mean Latency (s)": lat_lora},
}, orient="index")

table_trainable


,MRR@10,NDCG@10,Recall@10,Recall@100,Precision@10,Mean Latency (s)
Pre-trained,0.11048,0.31594,0.15499,0.31151,0.24334,0.007657
Full Fine-Tuning,0.11594,0.33163,0.16042,0.35197,0.26192,0.007628
LoRA,0.10705,0.30578,0.15590,0.29588,0.23560,0.010632


<div style="text-align: right; direction: rtl;">

**8.باتوجه به نتایج، در چه شرایطی تنظیم دقیق کامل و چه زمانی  peft را توصیه می کنید؟**   

با توجه به نتایج، Full Fine-Tuning زمانی توصیه می‌شود که حداکثر دقت (NDCG@10، Recall@100، Precision@10 بالاتر) هدف اصلی باشد و منابع محاسباتی و زمان آموزش محدودیت جدی نداشته باشند؛ چون در همه‌ی معیارهای کیفی بهتر عمل کرده و latency کمتری هم دارد.
در مقابل، PEFT/LoRA زمانی مناسب‌تر است که هزینه و منابع محدود باشند، بخواهید سریع‌تر به نتیجه برسید یا نیاز به به‌روزرسانی‌های مکرر دارید؛ هرچند افت اندکی در کیفیت نسبت به Full Fine-Tuning قابل انتظار است.

# <div style="text-align: center; direction: rtl;"><h1 align="center" style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>بخش ششم - رنک کردن Hybrid <br>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</h1></div>


In [111]:
def min_max_normalize(scores_dict):
    scores = list(scores_dict.values())
    if len(scores) == 0:
        return scores_dict

    min_s, max_s = min(scores), max(scores)

    if max_s == min_s:
        return {d: 0.0 for d in scores_dict}

    return {
        d: (s - min_s) / (max_s - min_s)
        for d, s in scores_dict.items()
    }


In [112]:
def hybrid_run_for_lambda(lambda_val, bm25_run, neural_run):
    hybrid_run = {}

    for qid in bm25_run:
        bm25_scores = min_max_normalize(bm25_run[qid])
        neural_scores = neural_run.get(qid, {})

        combined = {}
        for doc_id in bm25_scores:
            if doc_id in neural_scores:
                combined[doc_id] = (
                    lambda_val * neural_scores[doc_id]
                    + (1 - lambda_val) * bm25_scores[doc_id]
                )

        hybrid_run[qid] = dict(
            sorted(combined.items(), key=lambda x: x[1], reverse=True)
        )

    return hybrid_run


In [113]:
lambdas = [0.1, 0.3, 0.5, 0.7, 0.9]
lambda_to_ndcg10 = {}
best_neural_run = finetuned_run


for lam in lambdas:
    hybrid_run = hybrid_run_for_lambda(
        lam,
        bm25_run_test,
        best_neural_run
    )

    metrics = evaluator.evaluate(
        qrels_test,
        hybrid_run,
        k_values=[10]
    )

    lambda_to_ndcg10[lam] = metrics[0]["NDCG@10"]

In [114]:
best_lambda = max(lambda_to_ndcg10, key=lambda_to_ndcg10.get)

print("Best λ (maximizing NDCG@10):", best_lambda)
print("Best NDCG@10:", lambda_to_ndcg10[best_lambda])

Best λ (maximizing NDCG@10): 0.9
Best NDCG@10: 0.33357


<div style="text-align: right; direction: rtl;">

**9. چرا hybrid retrieval systems که روش های spars و dense را باهم ترکیب میکنند، معمولا بهتر از هرکدام به تنهایی عمل میکنند؟**

روش‌های sparse در تطابق دقیق واژگان، اصطلاحات تخصصی و کلمات نادر بسیار قوی‌اند و معمولاً اسناد مرتبط را به‌خوبی پوشش می‌دهند، اما درک معنایی محدودی دارند. در مقابل، روش‌های dense شباهت مفهومی و هم‌معنایی را حتی بدون تطابق لفظی تشخیص می‌دهند، ولی ممکن است اسناد واژگانیِ دقیق را از دست بدهند یا به موارد ظاهراً مرتبط امتیاز دهند. ترکیب این دو در سیستم‌های hybrid باعث می‌شود ضعف هرکدام توسط دیگری جبران شده و معیار های Recall و Precision بهتر شوند.

**10. مشکل vocabulary mismatch چیست و چگونه dense retrievers آن را حل میکند ؟**

مشکل Vocabulary Mismatch زمانی رخ می‌دهد که کوئری کاربر و سند مرتبط از نظر واژگانی یکسان نباشند، حتی اگر از نظر معنایی به یک مفهوم اشاره کنند؛ در این حالت روش‌های واژه‌محور مانند BM25 ممکن است سند مرتبط را بازیابی نکنند.
Dense retrievers با نمایش کوئری و اسناد در یک فضای برداری معنایی مشترک، شباهت مفهومی و هم‌معنایی را مدل می‌کنند و وابسته به تطابق دقیق کلمات نیستند، بنابراین می‌توانند اسنادی را بازیابی کنند که از واژگان متفاوت ولی با معنای یکسان یا نزدیک استفاده کرده‌اند.


# <div style="text-align: center; direction: rtl;"><h1 align="center" style="font-size: 24px; padding: 20px;">━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br> بخش هفتم - پیاده سازی یک RERANKER مبتنی بر LLM با استفاده از GPT-5.1-mini API <br>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</h1></div>


In [115]:
def get_top_k_docs(run, qid, k=20):
    return list(run[qid].keys())[:k]

In [116]:
from openai import OpenAI

client = OpenAI(
    api_key="aa-FdjrdReGv8Tpa9Oil1Vb8roI5JsC30TgMX23yTwiBVkOoSeG",  # با کلید واقعی خود جایگزین کنید
    base_url="https://api.avalai.ir/v1",  # آدرس پایه
)

In [122]:
import json

def llm_score_zero_shot_batch(query, doc_ids, corpus):
    docs_block = "\n".join(
        f"[{i+1}] ({doc_id}) {corpus[doc_id]}"
        for i, doc_id in enumerate(doc_ids)
    )

    prompt = f"""
    You are a medical information retrieval expert.

    Query:
    "{query}"

    Below are 20 candidate documents.
    For each document, assign a relevance score from 0 to 10.

    Documents:
    {docs_block}

    Output ONLY a JSON list in this exact format:
    [
      {{"doc_id": "<ID>", "score": <0-10>}}
    ]
    """

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return json.loads(response.choices[0].message.content)


In [141]:
def llm_score_cot_batch(query, doc_ids, corpus):
    docs_block = "\n".join(
        f"[{i+1}] ({doc_id}) {corpus[doc_id]}"
        for i, doc_id in enumerate(doc_ids)
    )

    prompt = f"""
    You are a medical information retrieval expert.

    Query:
    "{query}"

    Below are 20 candidate documents.
    For each document:
    - briefly explain whether and why the document is relevant to the query.
    - give a score from 0 to 10.

    Documents:
    {docs_block}

    Output format (repeat exactly for each document):
    DocID: <ID> (For each document, use the EXACT document ID shown in parentheses. Example: (DocID: MED-2434))
    Explanation: <one or two sentences>
    Score: <0-10>
    """

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    text = response.choices[0].message.content

    results = {}

    pattern = re.compile(
        r"DocID:\s*(MED-\d+)\s*"
        r"Explanation:\s*(.*?)\s*"
        r"Score:\s*(\d+)",
        re.S
    )


    for match in pattern.finditer(text):
        doc_id = match.group(1)
        explanation = match.group(2).strip()
        score = int(match.group(3))

        results[doc_id] = {
            "score": score,
            "explanation": explanation
        }

    return results

In [142]:
def llm_rerank_query_batch(
    qid,
    query_text,
    bi_encoder_run,
    corpus,
    scoring_fn_batch,
    top_k_candidates=20,
    final_k=10,
    keep_explanations=False
):
    doc_ids = get_top_k_docs(bi_encoder_run, qid, top_k_candidates)

    raw_scores = scoring_fn_batch(query_text, doc_ids, corpus)

    scores = {}

    if isinstance(raw_scores, list):
        for d in raw_scores:
            scores[d["doc_id"]] = {
                "score": int(d["score"]),
                "explanation": None
            }

    # حالت 2 یا 3: dict
    elif isinstance(raw_scores, dict):
        for doc_id, val in raw_scores.items():
            if isinstance(val, dict):
                scores[doc_id] = {
                    "score": int(val.get("score", 0)),
                    "explanation": val.get("explanation")
                }
            else:
                scores[doc_id] = {
                    "score": int(val),
                    "explanation": None
                }

    else:
        raise ValueError("Unsupported LLM scoring output format")

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1]["score"],
        reverse=True
    )

    if keep_explanations:
        return dict(ranked[:final_k])
    else:
        return {
            doc_id: info["score"]
            for doc_id, info in ranked[:final_k]
        }

In [143]:
subset_queries = list(queries_test.items())[:10]

llm_run_zero = {}
llm_run_cot = {}

for qid, qtext in subset_queries:
    llm_run_zero[qid] = llm_rerank_query_batch(
        qid, qtext, best_neural_run, corpus_texts,
        llm_score_zero_shot_batch,
        keep_explanations=False
    )

    llm_run_cot[qid] = llm_rerank_query_batch(
        qid, qtext, best_neural_run, corpus_texts,
        llm_score_cot_batch,
        keep_explanations=True
    )

In [144]:
evaluator.evaluate(qrels_test, llm_run_zero, k_values=[10])

({'NDCG@10': 0.40468},
 {'MAP@10': 0.07549},
 {'Recall@10': 0.09419},
 {'P@10': 0.37})

In [145]:
llm_run_cot_scores = {}

for qid, docs in llm_run_cot.items():
    llm_run_cot_scores[qid] = {
        doc_id: info["score"]
        for doc_id, info in docs.items()
    }
evaluator.evaluate(qrels_test, llm_run_cot_scores, k_values=[10])

({'NDCG@10': 0.29056},
 {'MAP@10': 0.05806},
 {'Recall@10': 0.06738},
 {'P@10': 0.27})

In [147]:
subset_200_queries = list(queries_test.items())[:200]

llm_run_zero_200 = {}

for qid, qtext in subset_200_queries:
    llm_run_zero_200[qid] = llm_rerank_query_batch(
        qid=qid,
        query_text=qtext,
        bi_encoder_run=best_neural_run,
        corpus=corpus_texts,
        scoring_fn_batch=llm_score_zero_shot_batch,
        top_k_candidates=20,
        final_k=10
    )


In [148]:
evaluator.evaluate(qrels_test, llm_run_zero_200, k_values=[10])

({'NDCG@10': 0.31}, {'MAP@10': 0.107}, {'Recall@10': 0.12877}, {'P@10': 0.239})

In [ ]:
# llm_run_cot_200 = {}

# for qid, qtext in subset_200_queries:
#     llm_run_cot_200[qid] = llm_rerank_query_batch(
#         qid=qid,
#         query_text=qtext,
#         bi_encoder_run=best_neural_run,
#         corpus=corpus_texts,
#         scoring_fn_batch=llm_score_cot_batch,
#         top_k_candidates=20,
#         final_k=10
#     )


<div style="text-align: right; direction: rtl;">

**11. LLM-based reranking و cross-encoder reranking را با هم مقایسه کنید، با تمرکز بر معیارهای: دقت، هزینه، تاخیر، قابلیت تفسیر، و سازگاری با حوزه های مختلف و سپس توضیح دهید که در چه شرایطی هر روش را ترجیح می دهید و چه معیارهایی در انتخاب بین آن ها مهم هستند.**

  1. **دقت :**    
    Cross-Encoder Reranking معمولاً دقت بالاتری دارد چون ورودیِ جفتِ (query, document) را به‌صورت مشترک می‌بیند و برای وظیفه‌ی رتبه‌بندی آموزش دیده است. در تمرین هم این مورد صدق می کند و NDCG@10  و Precision@10 Cross-Encoder Reranker از LLM-based Reranker بالاتر است.
  2. **هزینه :**
    Cross-encoder بسیار ارزان‌تر است و هزینه‌ی توکن + API؛ با افزایش طول اسناد و تعداد کوئری‌ها به‌سرعت گران می‌شود.
  3. **تاخیر :**
    Cross-Encoder سریعتر است اما LLM-Based کندتر بوده و به شبکه و مدل هم وابسته است.
  4. **قابلیت تفسیر :**
    Cross-Encoder چون صرفا یک امتیاز می دهد، چندان تفسیر پذیر نیست اما LLM-Based با با Chain-of-Thought می‌تواند توضیح متنی ارائه دهد
  5. **سازگاری با حوزه‌های مختلف :**
    Cross-Encoder برای دامنه های جدید معمولا نیاز به Fine-tuning دارد اما LLM ها به خوبی با دامنه های جدید سازگارند.

  اگر دقت پایدار، هزینه و تأخیر کم و استفاده در سیستم‌های آنلاین مهم است، Cross-Encoder انتخاب مناسب‌تری است.
اگر کوئری‌ها پیچیده یا چنددامنه‌ای هستند و تفسیرپذیری اهمیت دارد و هزینه/تاخیر قابل‌قبول است، LLM-based reranker ترجیح داده می‌شود.

**12. اگر پرس و جوها به فارسی و اسناد به انگلیسی باشند، چه رویکرد هایی پیشنهاد میکنید؟ مزایا و معایب ترجمه ماشینی و تعبیه چندزبانه را مقایسه کنید.**

  1. **رویکرد ترجمه ماشینی** : ترجمه پرس و جوی فارسی به انگلیسی (یا برعکس) و سپس بازیابی
      - **مزایا :**  
        - استفاده از مدل های بازیابی قدرتمند تک زبانه بخصوص به زبان انگلیسی

      - **معایب :**

        - احتمال دارد مدل تحت تاثیر خطای ترجمه قرار بگیرد و عملکرد بد مدل را نمی توان لزوما به خود مدل تعمیم داد.
        - ممکن است خطای ترجمه بخصوص در اصطلاحات تخصصی پزشکی رخ دهد.
        - یک هزینه محاسباتی مضاعفی برای ترجمه ماشینی متحمل می شود + هزینه زمانی
  2. **تعبیه های چندزبانی :** نگاشت پرس‌وجو و سندِ دو زبان مختلف به یک فضای برداری مشترک بطوریکه اگر دو متن از نظر معنا شبیه باشند، حتی با زبان‌های متفاوت، بردارهایشان به هم نزدیک می‌شود.
      - **مزایا :**
          - خطای ترجمه وارد سیستم نمی‌شود
          
          - درک بهتر شباهت معنایی بین زبان‌ها و قابلیت مقیاس پذیری بالا با زبان های مختلف

      - **معایب :**
          - دقت کمتری نسبت به مدل تک زبانه تخصصی دارد. بخصوص مدل های آموزش دیده روی زبان انگلیسی

**13. برای پرس و جوهایی که هیچ مثال مشابهی در training set ندارند، چه استراتژی هایی وجود دارند؟**

  1. **استفاده از مدل‌های ازپیش‌آموزش‌دیده :**    
    تکیه بر مدل‌هایی که روی داده‌های بزرگ و متنوع آموزش دیده‌اند (مثل bi-encoderهای عمومی یا LLMها) تا بدون نیاز به نمونه‌ی مشابه، شباهت معنایی را تشخیص دهند.
  2. **بازرتبه‌بندی با LLM یا Cross-Encoder :**     
    ابتدا بازیابی گسترده انجام می‌شود، سپس با مدل‌های قوی‌تر (cross-encoder یا LLM reranker) اسناد دقیق‌تر انتخاب می‌شوند
  3. **گسترش پرس‌وجو (Query Expansion / Reformulation) :**    
    افزودن مترادف‌ها، عبارات مرتبط یا بازنویسی پرس‌وجو (به کمک LLMها هم میتوان این کار را انجام داد) برای کاهش فاصله‌ی واژگانی و بهبود بازیابی اولیه.
  4. **بازیابی هیبریدی (Hybrid Retrieval) :**     
    ترکیب روش‌های sparse (مانند BM25) با dense، تا هم تطابق واژگانیِ دقیق پوشش داده شود و هم شباهت معنایی؛

